# Task
The objective of this project is to build a professional-grade Portfolio Stress Testing Engine. This engine will allow users to quantify potential losses in a diversified financial portfolio under both historical and hypothetical extreme market scenarios. We will utilize real-market data fetched via yfinance, apply advanced risk metrics like Value at Risk (VaR) and Expected Shortfall (ES), and create interactive dashboards to visualize tail risks. This project demonstrates skills in data engineering, quantitative finance, and data visualization, making it an ideal portfolio piece for risk management and data science roles.

# Portfolio Stress Testing Engine

## Project Overview
In the financial world, **Stress Testing** is a critical risk management process used to evaluate how a portfolio of assets would perform under extreme market conditions. Unlike standard risk metrics that look at average volatility, stress testing focuses on 'tail risks'—low-probability, high-impact events.

### Objective
This project aims to build a comprehensive, production-grade Stress Testing Engine. Key features include:
- **Real-time Data Integration**: Fetching live and historical market data using `yfinance`.
- **Historical Replays**: Simulating how current holdings would have fared during the 2008 Financial Crisis or the 2020 COVID-19 crash.
- **Hypothetical Scenarios**: Quantifying the impact of interest rate hikes or sector-specific crashes.
- **Advanced Analytics**: Calculating Value at Risk (VaR) and Expected Shortfall (ES).

### Why This Matters for Your Career
This project demonstrates proficiency in:
1. **Data Engineering**: Building robust data pipelines with error handling.
2. **Quantitative Finance**: Applying mathematical models to financial time series.
3. **Visual Analytics**: Using Plotly to communicate complex risk data to stakeholders.

**Reasoning**:
Installing the required libraries is the final technical part of the environment setup subtask.



In [13]:

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
print("Environment Setup Complete: Libraries installed and imported.")

Environment Setup Complete: Libraries installed and imported.


## Data Ingestion Pipeline

### Subtask:
Implement a robust script to fetch historical price data for a diversified portfolio using yfinance.


**Reasoning**:
I will define the tickers and use `yfinance` to download historical adjusted closing prices, then verify the download success.



In [1]:
import yfinance as yf
import pandas as pd


tickers = ['SPY', 'QQQ', 'BND', 'GLD', 'VTI']


start_date = '2008-01-01'
end_date = '2024-01-01'

print(f"Fetching data for: {tickers} from {start_date} to {end_date}...")

try:
   
    raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)

 
    if isinstance(raw_data.columns, pd.MultiIndex):
        portfolio_data = raw_data['Adj Close']
    else:
        portfolio_data = raw_data[['Adj Close']]

  
    downloaded_tickers = portfolio_data.columns.tolist()
    missing_tickers = set(tickers) - set(downloaded_tickers)

    if missing_tickers:
        print(f"Warning: Could not fetch data for {missing_tickers}")
    else:
        print("Successfully downloaded data for all tickers.")

   
    print(f"Data Shape: {portfolio_data.shape}")
    display(portfolio_data.head())

except Exception as e:
    print(f"An error occurred during data ingestion: {e}")

Fetching data for: ['SPY', 'QQQ', 'BND', 'GLD', 'VTI'] from 2008-01-01 to 2024-01-01...


[*********************100%***********************]  5 of 5 completed

Successfully downloaded data for all tickers.
Data Shape: (4027, 5)


Ticker,BND,GLD,QQQ,SPY,VTI
Date,,,,,
2008-01-02,43.724251,84.860001,43.356754,103.495598,51.553982
2008-01-03,43.780781,85.570000,43.537380,103.445633,51.492855
2008-01-04,43.831650,85.129997,41.628006,100.910530,50.116234
2008-01-07,43.842960,84.769997,41.430180,100.824821,50.141376
2008-01-08,43.888138,86.779999,40.355087,99.196701,49.242817


## Data Cleaning and Feature Engineering

### Subtask:
Clean the 'portfolio_data' and calculate daily returns for each asset.


**Reasoning**:
I will check for missing values, apply forward-fill as it's standard for financial time-series to avoid look-ahead bias, and then compute the daily returns.



In [9]:

missing_vals = portfolio_data.isnull().sum()
print("Missing values per ticker:\n", missing_vals)


portfolio_data_cleaned = portfolio_data.ffill()


returns_df = portfolio_data_cleaned.pct_change().dropna()

print("\nDaily Returns Shape:", returns_df.shape)
display(returns_df.head())


print("\nRemaining missing values in returns:", returns_df.isnull().sum().sum())

Missing values per ticker:
 Ticker
BND    0
GLD    0
QQQ    0
SPY    0
VTI    0
dtype: int64

Daily Returns Shape: (4026, 5)


Ticker,BND,GLD,QQQ,SPY,VTI
Date,,,,,
2008-01-03,0.001293,0.008367,0.004166,-0.000483,-0.001186
2008-01-04,0.001162,-0.005142,-0.043856,-0.024507,-0.026734
2008-01-07,0.000258,-0.004229,-0.004752,-0.000849,0.000502
2008-01-08,0.001030,0.023711,-0.025949,-0.016148,-0.017921
2008-01-09,-0.001159,-0.002650,0.021313,0.010510,0.010584



Remaining missing values in returns: 0


## Calculate Portfolio Performance Metrics

### Subtask:
Assume an equal-weighted portfolio and calculate the aggregate portfolio daily returns, cumulative returns, and basic volatility metrics.


**Reasoning**:
I will calculate the aggregate portfolio returns and volatility using the cleaned returns data.



In [ ]:
import numpy as np
num_assets = len(returns_df.columns)

weights = np.array([1/num_assets] * num_assets)


returns_df['Portfolio_Return'] = returns_df.iloc[:, :num_assets].dot(weights)


returns_df['Cumulative_Return'] = (1 + returns_df['Portfolio_Return']).cumprod()


ann_volatility = returns_df['Portfolio_Return'].std() * np.sqrt(252)

print(f"Equal-Weighted Portfolio Annualized Volatility: {ann_volatility:.2%}")
display(returns_df[['Portfolio_Return', 'Cumulative_Return']].head())
print("Portfolio metrics calculated.")

NameError: name 'np' is not defined

## Historical Stress Testing (Tail Risk)

### Subtask:
Analyze portfolio performance during specific historical crises (e.g., 2008 Financial Crisis, 2020 COVID-19 crash) and calculate Value at Risk (VaR) and Expected Shortfall (ES).


**Reasoning**:
I will define the historical crisis periods and implement the functions for Value at Risk (VaR) and Expected Shortfall (ES). Then, I will apply these to the portfolio's historical returns to evaluate performance during the 2008 and 2020 crashes.



In [17]:
# 1. Define date ranges for historical stress events
stress_periods = {
    '2008 Financial Crisis': ('2008-01-01', '2009-03-31'),
    '2020 COVID-19 Crash': ('2020-02-01', '2020-04-30')
}

# 2 & 3. Functions for Historical VaR and ES
def calculate_historical_var(returns, confidence_level=0.95):
    """Calculates the Historical Value at Risk (VaR)."""
    return np.percentile(returns, (1 - confidence_level) * 100)

def calculate_historical_es(returns, confidence_level=0.95):
    """Calculates the Historical Expected Shortfall (ES)."""
    var = calculate_historical_var(returns, confidence_level)
    return returns[returns <= var].mean()

def calculate_max_drawdown(cumulative_returns):
    """Calculates the maximum drawdown from a cumulative return series."""
    peak = cumulative_returns.cummax()
    drawdown = (cumulative_returns - peak) / peak
    return drawdown.min()

# 4 & 5. Analyze performance during stress periods and compare with overall period
results = []

# Overall period metrics
overall_returns = returns_df['Portfolio_Return']
results.append({
    'Period': 'Overall (2008-2023)',
    'VaR (95%)': calculate_historical_var(overall_returns),
    'ES (95%)': calculate_historical_es(overall_returns),
    'Max Drawdown': calculate_max_drawdown(returns_df['Cumulative_Return'])
})

for name, (start, end) in stress_periods.items():
    period_data = returns_df.loc[start:end].copy()
    if not period_data.empty:
        # Re-calculate cumulative return starting from 1 for just this period for MDD
        period_data['Period_Cum_Return'] = (1 + period_data['Portfolio_Return']).cumprod()

        results.append({
            'Period': name,
            'VaR (95%)': calculate_historical_var(period_data['Portfolio_Return']),
            'ES (95%)': calculate_historical_es(period_data['Portfolio_Return']),
            'Max Drawdown': calculate_max_drawdown(period_data['Period_Cum_Return'])
        })

# Display Results
results_df = pd.DataFrame(results)
print("Historical Stress Testing Results:")
display(results_df)

# Print interpretation
print("\nKey Insights:")
for index, row in results_df.iterrows():
    if row['Period'] != 'Overall (2008-2023)':
        print(f"- During the {row['Period']}, the Expected Shortfall (ES) was {row['ES (95%)']:.2%}, significantly worse than the overall average.")

Historical Stress Testing Results:


,Period,VaR (95%),ES (95%),Max Drawdown
0,Overall (2008-2023),-0.012162,-0.019799,-0.333591
1,2008 Financial Crisis,-0.024700,-0.035495,-0.333591
2,2020 COVID-19 Crash,-0.036375,-0.056696,-0.212430



Key Insights:
- During the 2008 Financial Crisis, the Expected Shortfall (ES) was -3.55%, significantly worse than the overall average.
- During the 2020 COVID-19 Crash, the Expected Shortfall (ES) was -5.67%, significantly worse than the overall average.


## Visual Risk Dashboard

### Subtask:
Create interactive visualizations using Plotly to show historical drawdowns, return distributions, and a comparison of performance across different stress scenarios.


**Reasoning**:
I will create a series of interactive plots using Plotly as per the instructions: a line chart for drawdowns, a histogram for return distribution with risk markers, and a bar chart comparing metrics across crisis periods.



In [18]:
import plotly.graph_objects as go
import plotly.express as px

# 1. Calculate historical drawdown series
peak = returns_df['Cumulative_Return'].cummax()
drawdown = (returns_df['Cumulative_Return'] - peak) / peak

# 2. Line chart showing Portfolio Drawdown
fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(x=drawdown.index, y=drawdown, fill='tozeroy', name='Drawdown', line=dict(color='red')))
fig_dd.update_layout(title='Historical Portfolio Drawdown (2008-2023)', xaxis_title='Date', yaxis_title='Drawdown (%)', template='plotly_white')
fig_dd.show()

# 3. Histogram of returns with VaR and ES markers
overall_var = results_df.loc[0, 'VaR (95%)']
overall_es = results_df.loc[0, 'ES (95%)']

fig_hist = px.histogram(returns_df, x='Portfolio_Return', nbins=100, title='Portfolio Daily Returns Distribution', labels={'Portfolio_Return': 'Daily Return'})
fig_hist.add_vline(x=overall_var, line_dash="dash", line_color="orange", annotation_text=f"VaR (95%): {overall_var:.2%}")
fig_hist.add_vline(x=overall_es, line_dash="dash", line_color="red", annotation_text=f"ES (95%): {overall_es:.2%}")
fig_hist.update_layout(template='plotly_white')
fig_hist.show()

# 4. Comparison Bar Chart
# Melting results_df for easier plotting with plotly express
comparison_df = results_df.melt(id_vars='Period', var_name='Metric', value_name='Value')

fig_bar = px.bar(comparison_df, x='Period', y='Value', color='Metric', barmode='group',
                 title='Risk Metric Comparison Across Stress Scenarios',
                 labels={'Value': 'Percentage Impact'})
fig_bar.update_layout(yaxis_tickformat='.1%', template='plotly_white')
fig_bar.show()

print("Visual Risk Dashboard generated successfully.")

Visual Risk Dashboard generated successfully.


## Final Task

### Subtask:
Summarize the findings from the stress tests and provide insights into the portfolio's resilience against market shocks.


## Implement Hypothetical Stress Scenarios

### Subtask:
Develop a module to simulate 'What-If' scenarios and quantify their impact on the portfolio.


**Reasoning**:
I will define the hypothetical stress scenarios, create a simulation function to apply these shocks to the mean historical returns, and then calculate the projected portfolio impact.



In [19]:
import pandas as pd
import numpy as np

# 1. Define hypothetical scenarios with delta-shocks
# Note: These are applied relative to the mean daily returns or as absolute loss events
hypothetical_scenarios = {
    'Interest Rate Hike': {
        'SPY': -0.02, 'QQQ': -0.03, 'BND': -0.05, 'GLD': -0.01, 'VTI': -0.02
    },
    'Tech Crash (with Flight to Quality)': {
        'SPY': -0.05, 'QQQ': -0.10, 'BND': 0.02, 'GLD': 0.03, 'VTI': -0.05
    }
}

# 2. Function to simulate scenario impact
def simulate_scenario(shocks, weights, asset_names):
    # Convert shocks to a vector aligned with the weights
    shock_vector = np.array([shocks.get(asset, 0) for asset in asset_names])
    # Calculate weighted impact
    portfolio_impact = np.dot(shock_vector, weights)
    return portfolio_impact

# 3. Calculate impacts
asset_names = ['BND', 'GLD', 'QQQ', 'SPY', 'VTI'] # Consistent order from returns_df
# Ensure weights are aligned with this order
weights_array = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

hypo_results = []
for scenario_name, shocks in hypothetical_scenarios.items():
    impact = simulate_scenario(shocks, weights_array, asset_names)
    hypo_results.append({
        'Scenario': scenario_name,
        'Projected Impact': impact
    })

# 4. Create and display DataFrame
hypothetical_results_df = pd.DataFrame(hypo_results)
print("Hypothetical Stress Scenario Impacts:")
display(hypothetical_results_df)

# Comparison with historical ES for context
print(f"\nFor context, the 2020 COVID Crash ES (95%) was: {results_df.loc[2, 'ES (95%)']:.2%}")

Hypothetical Stress Scenario Impacts:


,Scenario,Projected Impact
0,Interest Rate Hike,-0.026
1,Tech Crash (with Flight to Quality),-0.030



For context, the 2020 COVID Crash ES (95%) was: -5.67%


## Create Stress Impact Waterfall Chart

### Subtask:
Generate a Plotly waterfall chart that visualizes the step-by-step contribution of individual assets to the total portfolio loss during a selected stress event.


**Reasoning**:
I will calculate the individual asset contributions for the 'Tech Crash' scenario by multiplying the predefined weights by the shocks, and then use Plotly's Waterfall graph object to visualize how each asset contributes to the total portfolio impact.



In [20]:
import plotly.graph_objects as go

# 1. Select the scenario and data
scenario_to_plot = 'Tech Crash (with Flight to Quality)'
shocks = hypothetical_scenarios[scenario_to_plot]
# asset_names and weights_array are already defined from previous cells

# 2. Calculate contributions: weight * shock
contributions = []
for i, asset in enumerate(asset_names):
    contribution = weights_array[i] * shocks.get(asset, 0)
    contributions.append(contribution)

total_impact = sum(contributions)

# 3. Prepare data for Plotly Waterfall
# Labels for the x-axis
x_labels = asset_names + ['Total Portfolio Impact']
# Measures define if the value is relative (contribution) or absolute (total)
measures = ['relative'] * len(asset_names) + ['total']
# Values for the bars
y_values = contributions + [total_impact]

# 4. Create the Waterfall Chart
fig_wf = go.Figure(go.Waterfall(
    name="Contribution",
    orientation="v",
    measure=measures,
    x=x_labels,
    textposition="outside",
    text=[f"{val:.2%}" for val in y_values],
    y=y_values,
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    increasing={"marker": {"color": "#2ca02c"}}, # Green for positive (hedges)
    decreasing={"marker": {"color": "#d62728"}}, # Red for negative (losses)
    totals={"marker": {"color": "#1f77b4"}}     # Blue for total
))

fig_wf.update_layout(
    title=f"Asset Contribution to Portfolio Impact: {scenario_to_plot}",
    yaxis_title="Impact (%)",
    showlegend=False,
    template='plotly_white',
    yaxis_tickformat='.1%'
)

fig_wf.show()

print(f"Waterfall chart generated for scenario: {scenario_to_plot}")

Waterfall chart generated for scenario: Tech Crash (with Flight to Quality)


## Calculate Advanced Risk-Adjusted Metrics

### Subtask:
Compute Sharpe Ratio and Sortino Ratio specifically for the historical stress periods and the overall timeframe.


**Reasoning**:
I will calculate the Sharpe Ratio and Sortino Ratio for the overall period and the historical stress periods to evaluate risk-adjusted performance during market shocks.



In [21]:
def calculate_sharpe_ratio(returns, rf=0, periods=252):
    """Calculates the annualized Sharpe Ratio."""
    excess_returns = returns - rf
    return np.mean(excess_returns) / np.std(excess_returns) * np.sqrt(periods)

def calculate_sortino_ratio(returns, rf=0, periods=252):
    """Calculates the annualized Sortino Ratio."""
    excess_returns = returns - rf
    downside_returns = excess_returns[excess_returns < 0]
    downside_std = np.std(downside_returns)
    return np.mean(excess_returns) / downside_std * np.sqrt(periods)

# Initialize list for metrics
risk_adjusted_metrics = []

# 1. Overall Metrics
risk_adjusted_metrics.append({
    'Period': 'Overall (2008-2023)',
    'Sharpe Ratio': calculate_sharpe_ratio(returns_df['Portfolio_Return']),
    'Sortino Ratio': calculate_sortino_ratio(returns_df['Portfolio_Return'])
})

# 2. Crisis Period Metrics
for name, (start, end) in stress_periods.items():
    period_returns = returns_df.loc[start:end, 'Portfolio_Return']
    if not period_returns.empty:
        risk_adjusted_metrics.append({
            'Period': name,
            'Sharpe Ratio': calculate_sharpe_ratio(period_returns),
            'Sortino Ratio': calculate_sortino_ratio(period_returns)
        })

# Display Results
risk_metrics_df = pd.DataFrame(risk_adjusted_metrics)
print("Advanced Risk-Adjusted Metrics:")
display(risk_metrics_df)

print("Note: A Sortino Ratio significantly lower than the Sharpe Ratio during crises indicates high downside volatility relative to total volatility.")

Advanced Risk-Adjusted Metrics:


,Period,Sharpe Ratio,Sortino Ratio
0,Overall (2008-2023),0.743035,0.937175
1,2008 Financial Crisis,-0.756986,-1.114438
2,2020 COVID-19 Crash,0.069118,0.089294


Note: A Sortino Ratio significantly lower than the Sharpe Ratio during crises indicates high downside volatility relative to total volatility.


## Final Project Packaging and Documentation

### Subtask:

Consolidate the notebook with architectural descriptions, professional code comments, and a final summary suitable for a portfolio .

# Project Conclusion & Architectural Overview

### Engine Architecture

This **Portfolio Stress Testing Engine** was designed with a modular architecture to ensure scalability and professional-grade analysis:

1. **Data Ingestion Layer**: Leverages the `yfinance` API to fetch high-fidelity historical adjusted closing prices for a diversified multi-asset portfolio.

2. **Quantitative Analytics Engine**: Implements custom statistical modules to calculate **Value at Risk (VaR)** and **Expected Shortfall (ES)** at a 95% confidence interval, alongside annualized **Sharpe** and **Sortino** ratios.

3. **Stress Simulation Module**: Features a 'What-If' framework to simulate hypothetical shocks, such as tech-specific crashes and interest rate hikes, allowing for proactive risk management.

4. **Visualization Layer**: Utilizes `Plotly` for interactive risk dashboards, including historical drawdown profiles, return distribution histograms, and contribution waterfall charts.

### Summary of Stress Testing Results

The analysis revealed significant differences in portfolio resilience across market regimes:
- **Historical Crises**: The **2020 COVID-19 Crash** exhibited the highest intensity tail risk, with an Expected Shortfall (ES) of **-5.67%**, significantly steeper than the **-3.55%** ES observed during the **2008 Financial Crisis**. However, the portfolio showed better risk-adjusted recovery in 2020 (Sortino: 0.09) compared to the sustained destruction of 2008 (Sortino: -1.11).

- **Hypothetical Scenarios**: Our simulated **'Tech Crash'** projected a total portfolio loss of **3.00%**. The waterfall analysis highlighted that the 'Flight to Quality' assets (BND and GLD) provided a combined **1.00% hedge**, partially offsetting the **4.00% loss** contributed by equity components (QQQ, SPY, VTI).

### Strategic Insights & Next Steps

- **Resilience**: The equal-weighted diversification into Gold (GLD) and Bonds (BND) acted as a critical stabilizer during equity-centric shocks.

- **Enhancements**: Future iterations of this engine could integrate **Monte Carlo Simulations** for probabilistic path forecasting and **Macroeconomic Factor Models** to link portfolio performance to inflation or GDP growth data.


## Project Q&A and Clarifications

### Subtask:
Address specific user questions regarding the project's methodology, metrics, or implementation details.


### Methodology Deep-Dive: Metrics & Simulations

To ensure this engine meets professional standards, it is important to understand the quantitative logic behind our risk assessments.

#### 1. VaR vs. Expected Shortfall (ES)
While both are measures of tail risk, they provide different insights:
- **Value at Risk (VaR 95%)**: This tells us the minimum loss we expect to see 5% of the time. For our portfolio, the historical VaR was approximately **-1.22%**. This means there is a 5% chance the portfolio will lose *at least* this much on any given day.
- **Expected Shortfall (ES 95%)**: Also known as 'Conditional VaR', ES goes further by answering: *'If we are in that worst 5% of cases, what is the average loss?'* In our results, the ES for the 2020 crash was **-5.67%**, which is significantly more severe than the VaR. ES is generally considered a superior metric for stress testing because it accounts for the severity of the 'tail' losses.

#### 2. 'What-If' Simulation Logic
The hypothetical stress scenarios use a **Linear Shock Model**:
- **Input**: We define a vector of percentage shocks ($S$) for each asset class (e.g., Tech -10%, Bonds +2%).
- **Calculation**: The total portfolio impact is the dot product of the asset weights ($W$) and the shocks ($S$).
  $$\text{Portfolio Impact} = \sum (W_i \times S_i)$$
- **Interpretation**: This allows us to see how diversification (e.g., holding non-correlated assets like GLD or BND) acts as a buffer during specific sector crashes.

#### 3. Interpreting the Dashboard
- **Drawdown Charts**: Visualize the 'pain' over time, showing how long it took for the portfolio to recover from its historical peaks.
- **Waterfall Charts**: Deconstruct the total loss into individual asset contributions, making it easy to identify which 'engine' failed and which 'anchor' held firm during a crisis.